# AIOps Silver transform stream

Reads PASS/WARN Bronze records from `aiops_bronze_vehicle_positions_validated_stream`, transforms them to Silver, and writes `aiops_silver_vehicle_positions_cleaned` for Silver AIOps scoring.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "hsl"

AIOPS_BRONZE_VALIDATED_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.aiops_bronze_vehicle_positions_validated_stream"
AIOPS_SILVER_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.aiops_silver_vehicle_positions_cleaned"
AIOPS_SILVER_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/silver/cleaned/hsl_vehicle_position"
CHECKPOINT_AIOPS_SILVER_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/silver/checkpoints/silver_transform"

TRIGGER_INTERVAL = "10 seconds"
WATERMARK_DELAY = "10 minutes"

RESET_TABLE = False
RESET_CHECKPOINT = False

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

for q in spark.streams.active:
    if q.name == "aiops_silver_vehicle_positions_cleaned":
        q.stop()

if RESET_TABLE:
    spark.sql(f"DROP TABLE IF EXISTS {AIOPS_SILVER_TABLE}")
    dbutils.fs.rm(AIOPS_SILVER_PATH, True)

if RESET_CHECKPOINT:
    dbutils.fs.rm(CHECKPOINT_AIOPS_SILVER_PATH, True)


In [ ]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {AIOPS_SILVER_TABLE} (
    topic STRING, partition INT, offset BIGINT, eventhub_enqueued_ts TIMESTAMP, message_key STRING, raw_json STRING,
    bronze_ingest_ts TIMESTAMP, ingest_date DATE, parse_ok BOOLEAN, parse_error STRING, source STRING, producer_ingest_ts_utc STRING,
    mqtt_topic STRING, mqtt_qos INT, mqtt_retain BOOLEAN,
    mqtt_host STRING, mqtt_port INT, topic_event_type STRING, topic_transport_mode STRING, event_type STRING, transport_mode STRING,
    topic_operator_id STRING, topic_vehicle_number STRING, topic_vehicle_number_norm STRING, topic_operator_id_norm STRING, topic_route_id STRING,
    topic_direction_id STRING, topic_headsign STRING, topic_start_time STRING, topic_next_stop_id STRING, desi STRING, dir STRING,
    operator_id STRING, vehicle_number STRING, event_ts_raw STRING, event_ts TIMESTAMP, event_ts_unix BIGINT, speed DOUBLE, heading INT,
    latitude DOUBLE, longitude DOUBLE, acceleration DOUBLE, delay_sec INT, odometer_m INT, door_status INT, operating_day DATE, journey_id INT,
    line_id STRING, journey_start_time STRING, location_source STRING, stop_id STRING, payload_route_id STRING, occupancy INT, route_id STRING,
    direction_id STRING, vehicle_id STRING, business_key STRING, silver_ingest_ts TIMESTAMP, dedup_event_ts TIMESTAMP, silver_event_date DATE,
    event_to_enqueue_delay_sec BIGINT, enqueue_to_bronze_delay_sec BIGINT, bronze_to_silver_delay_sec BIGINT, event_to_silver_delay_sec BIGINT
)
USING DELTA
PARTITIONED BY (silver_event_date)
LOCATION "{AIOPS_SILVER_PATH}"
''')


In [ ]:
def normalize_str(col_expr):
    return (
        F.when(col_expr.isNull(), None)
        .otherwise(F.regexp_replace(F.trim(col_expr.cast("string")), r"^0+(?=\d)", ""))
    )


envelope_schema = T.StructType([
    T.StructField("ingest_ts_utc", T.StringType(), True),
    T.StructField("source", T.StringType(), True),
    T.StructField("mqtt", T.StructType([
        T.StructField("host", T.StringType(), True),
        T.StructField("port", T.IntegerType(), True),
        T.StructField("topic", T.StringType(), True),
        T.StructField("qos", T.IntegerType(), True),
        T.StructField("retain", T.BooleanType(), True),
    ]), True),
    T.StructField("topic_parsed", T.StructType([
        T.StructField("event_type", T.StringType(), True),
        T.StructField("transport_mode", T.StringType(), True),
        T.StructField("operator_id", T.StringType(), True),
        T.StructField("vehicle_number", T.StringType(), True),
        T.StructField("route_id", T.StringType(), True),
        T.StructField("direction_id", T.StringType(), True),
        T.StructField("headsign", T.StringType(), True),
        T.StructField("start_time", T.StringType(), True),
        T.StructField("next_stop_id", T.StringType(), True),
    ]), True),
    T.StructField("payload", T.StructType([
        T.StructField("VP", T.StructType([
            T.StructField("desi", T.StringType(), True),
            T.StructField("dir", T.StringType(), True),
            T.StructField("oper", T.IntegerType(), True),
            T.StructField("veh", T.IntegerType(), True),
            T.StructField("tst", T.StringType(), True),
            T.StructField("tsi", T.LongType(), True),
            T.StructField("spd", T.DoubleType(), True),
            T.StructField("hdg", T.IntegerType(), True),
            T.StructField("lat", T.DoubleType(), True),
            T.StructField("long", T.DoubleType(), True),
            T.StructField("acc", T.DoubleType(), True),
            T.StructField("dl", T.IntegerType(), True),
            T.StructField("odo", T.IntegerType(), True),
            T.StructField("drst", T.IntegerType(), True),
            T.StructField("oday", T.StringType(), True),
            T.StructField("jrn", T.IntegerType(), True),
            T.StructField("line", T.IntegerType(), True),
            T.StructField("start", T.StringType(), True),
            T.StructField("loc", T.StringType(), True),
            T.StructField("stop", T.IntegerType(), True),
            T.StructField("route", T.StringType(), True),
            T.StructField("occu", T.IntegerType(), True),
        ]), True),
    ]), True),
])

bronze_stream_df = spark.readStream.table(AIOPS_BRONZE_VALIDATED_TABLE)

parsed_df = (
    bronze_stream_df
    .withColumn("envelope", F.from_json(F.col("raw_json"), envelope_schema))
    .withColumn("vp", F.col("envelope.payload.VP"))
)

silver_clean_df = (
    parsed_df
    .select(
        "topic", "partition", "offset", "eventhub_enqueued_ts", "message_key", "raw_json",
        "bronze_ingest_ts", "ingest_date", "parse_ok", "parse_error", "source",
        "producer_ingest_ts_utc", "mqtt_topic", "mqtt_qos", "mqtt_retain",
        F.col("envelope.mqtt.host").alias("mqtt_host"),
        F.col("envelope.mqtt.port").alias("mqtt_port"),
        F.col("envelope.topic_parsed.event_type").alias("topic_event_type"),
        F.col("envelope.topic_parsed.transport_mode").alias("topic_transport_mode"),
        F.col("event_type"),
        F.col("transport_mode"),
        F.col("envelope.topic_parsed.operator_id").alias("topic_operator_id"),
        F.col("envelope.topic_parsed.vehicle_number").alias("topic_vehicle_number"),
        F.col("envelope.topic_parsed.route_id").alias("topic_route_id"),
        F.col("envelope.topic_parsed.direction_id").alias("topic_direction_id"),
        F.col("envelope.topic_parsed.headsign").alias("topic_headsign"),
        F.col("envelope.topic_parsed.start_time").alias("topic_start_time"),
        F.col("envelope.topic_parsed.next_stop_id").alias("topic_next_stop_id"),
        F.col("vp.desi").alias("desi"),
        F.col("vp.dir").alias("dir"),
        F.col("vp.oper").cast("string").alias("operator_id"),
        F.col("vp.veh").cast("string").alias("vehicle_number"),
        F.col("vp.tst").alias("event_ts_raw"),
        F.to_timestamp("vp.tst").alias("event_ts"),
        F.col("vp.tsi").cast("long").alias("event_ts_unix"),
        F.col("vp.spd").cast("double").alias("speed"),
        F.col("vp.hdg").cast("int").alias("heading"),
        F.col("vp.lat").cast("double").alias("latitude"),
        F.col("vp.long").cast("double").alias("longitude"),
        F.col("vp.acc").cast("double").alias("acceleration"),
        F.col("vp.dl").cast("int").alias("delay_sec"),
        F.col("vp.odo").cast("int").alias("odometer_m"),
        F.col("vp.drst").cast("int").alias("door_status"),
        F.to_date("vp.oday").alias("operating_day"),
        F.col("vp.jrn").cast("int").alias("journey_id"),
        F.col("vp.line").cast("string").alias("line_id"),
        F.col("vp.start").alias("journey_start_time"),
        F.col("vp.loc").alias("location_source"),
        F.col("vp.stop").cast("string").alias("stop_id"),
        F.col("vp.route").alias("payload_route_id"),
        F.col("vp.occu").cast("int").alias("occupancy"),
    )
    .withColumn("topic_vehicle_number_norm", normalize_str(F.col("topic_vehicle_number")))
    .withColumn("topic_operator_id_norm", normalize_str(F.col("topic_operator_id")))
    .withColumn("route_id", F.coalesce(F.col("payload_route_id"), F.col("topic_route_id"), F.col("desi")))
    .withColumn("direction_id", F.coalesce(F.col("dir"), F.col("topic_direction_id")))
    .withColumn("vehicle_id", F.concat_ws("_", F.coalesce(F.col("operator_id"), F.col("topic_operator_id_norm")), F.coalesce(F.col("vehicle_number"), F.col("topic_vehicle_number_norm"))))
    .withColumn("business_key", F.concat_ws("|", F.coalesce(F.col("vehicle_id"), F.lit("")), F.coalesce(F.col("event_ts_unix").cast("string"), F.lit("")), F.coalesce(F.col("route_id"), F.lit("")), F.coalesce(F.col("direction_id"), F.lit(""))))
    .withColumn("silver_ingest_ts", F.current_timestamp())
    .withColumn("dedup_event_ts", F.coalesce(F.col("event_ts"), F.col("eventhub_enqueued_ts"), F.col("bronze_ingest_ts"), F.current_timestamp()))
    .withColumn("silver_event_date", F.to_date(F.coalesce(F.col("event_ts"), F.col("eventhub_enqueued_ts"), F.col("bronze_ingest_ts"), F.current_timestamp())))
    .withColumn("event_to_enqueue_delay_sec", (F.col("eventhub_enqueued_ts").cast("long") - F.col("event_ts").cast("long")).cast("long"))
    .withColumn("enqueue_to_bronze_delay_sec", (F.col("bronze_ingest_ts").cast("long") - F.col("eventhub_enqueued_ts").cast("long")).cast("long"))
    .withColumn("bronze_to_silver_delay_sec", (F.col("silver_ingest_ts").cast("long") - F.col("bronze_ingest_ts").cast("long")).cast("long"))
    .withColumn("event_to_silver_delay_sec", (F.col("silver_ingest_ts").cast("long") - F.col("event_ts").cast("long")).cast("long"))
)

silver_dedup_df = silver_clean_df.withWatermark("dedup_event_ts", WATERMARK_DELAY).dropDuplicates(["business_key"])

aiops_silver_query = (
    silver_dedup_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_AIOPS_SILVER_PATH)
    .queryName("aiops_silver_vehicle_positions_cleaned")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .toTable(AIOPS_SILVER_TABLE)
)

print("AIOps Silver stream started.")
print("  Query name:", aiops_silver_query.name)
print("  Query ID  :", aiops_silver_query.id)
